[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/46.%20The%20Competitive%20Port%20Pricing%20Problem/P46-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 46. The Competitive Port Pricing Problem

## Tier 7: The Human–AI Partnership

### Goal

This notebook demonstrates a human–AI collaboration process for port pricing. AI systems produce explainable price recommendations, human managers contribute strategic considerations, and a consensus rule combines the recommendations into a price. A logit choice model estimates the resulting allocation of shipping-line demand across ports. A simulation then records profits, trust, and decision outcomes.

The notebook is a transparent, rule-based simulation. It is not an empirical study, a trained AI system, or an optimization model that proves a globally optimal price. Its numerical outputs depend on the stated assumptions and, where random choices are used, on the unseeded random draws.

### Inputs, outputs, and scope

**Inputs:** port costs and capacities, shipping-line demands and relationship attributes, market price-sensitivity parameter, and price bounds. The example also defines manager and AI-system characteristics.

**Outputs:** AI recommendations and explanations, human strategic insights, collaborative prices, estimated market shares and profits, simulation summaries, and plots.

**Model assumptions:** demand for each shipping line is allocated among the modeled ports using a price-only logit choice model. The code treats a shipping line's demand as fully allocated across those ports. Port capacities are reported but are not imposed as constraints; relationship strength and strategic importance are displayed but do not directly enter the demand or profit equations. Human adjustments, confidence scores, trust changes, and partnership-impact percentages are illustrative rules, not estimated causal effects. The model excludes competitor responses, operating dynamics beyond the stated demand allocation, and uncertainty in demand.

### Collaboration process

1. AI systems analyze a pricing situation and produce recommendations with explanations.
2. Human managers provide strategic insights and assess the AI recommendations.
3. The consensus rule computes a confidence-weighted average of AI prices and adds the rule-based human strategic adjustment.
4. The collaborative price is bounded by the market's minimum and maximum prices.
5. The logit model estimates demand allocation; profit and other outcomes are calculated from that allocation.
6. The simulation records outcomes and updates the illustrative trust and learning measures.

### Mathematical model

For shipping line $j$, let $p_{ij}$ be the price charged by port $i$, and let $\beta$ be price sensitivity. The port's price utility is $u_{ij}=\exp(-\beta p_{ij})$. The predicted demand allocated to port $i$ is

$$q_{ij}=D_j\frac{\exp(-\beta p_{ij})}{\sum_{k=1}^{m}\exp(-\beta p_{kj})},$$

where $D_j$ is shipping line $j$'s total demand and $m$ is the number of ports. For finite prices and positive demand, the shares across ports sum to $D_j$. This formulation assumes price is the only modeled factor in port choice.

For a collaborative decision, the AI component is the confidence-weighted mean

$$\bar p^{\,AI}=\frac{\sum_{a=1}^{A} c_a p_a}{\sum_{a=1}^{A}c_a},$$

when the total AI confidence is positive; otherwise the code uses the arithmetic mean of the AI prices. Here $p_a$ and $c_a$ are AI system $a$'s recommended price and confidence, and $A$ is the number of AI systems. The human adjustment $h$ is computed from the strategic-factor rules in the code. The final price is

$$p^{\mathrm{collab}}=\min\!\left(p_{\max},\max\left(p_{\min},\bar p^{\,AI}+h\right)\right).$$

If port $i$ has unit cost $c_{ij}$ for shipping line $j$, its modeled profit and profit margin are

$$\pi_{ij}=(p_{ij}-c_{ij})q_{ij},\qquad
M_{ij}=\frac{p_{ij}-c_{ij}}{p_{ij}}\quad\text{for }p_{ij}>0.$$

The implementation corresponds to `calculate_market_share`, `_reach_consensus`, and `_calculate_decision_outcomes`. These equations describe the rules being simulated; they do not define an optimization problem or establish that the resulting prices maximize profit.

### Reading results responsibly

A simulated profit is the outcome of the stated price, cost, and logit allocation assumptions. It is not evidence that collaboration causes a profit improvement. In the negotiation illustration, partnership-impact percentages are scenario assumptions applied to reported margins; they are not generated by the demand model. Comparisons between approaches therefore illustrate the consequences of those specified rules rather than independently measured performance.

In [ ]:
import random
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Port:
    """A port with a capacity and a cost for each shipping line."""

    id: int
    name: str
    capacity: float
    costs: List[float]


@dataclass
class ShippingLine:
    """A shipping line with demand and relationship characteristics."""

    id: int
    name: str
    demand: float
    priority: int
    relationship_strength: float = 0.5
    strategic_importance: float = 0.5


@dataclass
class MarketParameters:
    """Market-wide price sensitivity and allowable price bounds."""

    price_sensitivity: float
    min_price: float
    max_price: float


class DecisionType(Enum):
    """Decision categories available to the partnership."""

    PRICING_DECISION = "pricing_decision"
    STRATEGIC_PLANNING = "strategic_planning"
    NEGOTIATION_SUPPORT = "negotiation_support"
    RISK_ASSESSMENT = "risk_assessment"
    MARKET_ANALYSIS = "market_analysis"


class ConfidenceLevel(Enum):
    """Illustrative confidence levels for recommendations."""

    VERY_LOW = 0.2
    LOW = 0.4
    MEDIUM = 0.6
    HIGH = 0.8
    VERY_HIGH = 1.0


@dataclass
class HumanManager:
    """Human manager whose rules represent strategic judgment."""

    id: int
    name: str
    experience_years: int
    expertise_areas: List[str]
    decision_style: str
    trust_in_ai: float = 0.5
    learning_rate: float = 0.1

    def provide_strategic_insight(self, situation: Dict) -> Dict:
        """Return illustrative strategic factors for the given customer."""
        insight = {
            "strategic_factors": [],
            "relationship_considerations": [],
            "risk_assessment": "medium",
            "confidence": 0.7,
            "reasoning": [],
        }

        shipping_line_id = situation.get("shipping_line_id")
        if shipping_line_id == 0:
            insight["strategic_factors"].append(
                "High-value customer requiring special attention"
            )
            insight["relationship_considerations"].append(
                "Long-term partnership potential"
            )
            insight["confidence"] = 0.8
        elif shipping_line_id == 1:
            insight["strategic_factors"].append(
                "Steady customer with growth potential"
            )
            insight["relationship_considerations"].append(
                "Maintain competitive pricing"
            )
            insight["confidence"] = 0.6
        elif shipping_line_id is not None:
            insight["strategic_factors"].append("Opportunistic business")
            insight["relationship_considerations"].append(
                "Price optimization priority"
            )
            insight["confidence"] = 0.5

        if self.experience_years > 10:
            insight["reasoning"].append(
                "Based on extensive market experience"
            )
            insight["confidence"] = min(1.0, insight["confidence"] + 0.1)

        if self.decision_style == "conservative":
            insight["risk_assessment"] = "high"
        elif self.decision_style == "aggressive":
            insight["risk_assessment"] = "low"

        return insight

    def evaluate_ai_recommendation(self, ai_recommendation: Dict) -> Dict:
        """Compare AI confidence with an illustrative human assessment."""
        ai_confidence = ai_recommendation.get("confidence", 0.5)
        human_confidence = 0.7
        alignment = 1.0 - abs(ai_confidence - human_confidence)
        evaluation = {
            "acceptance_level": "neutral",
            "confidence_alignment": alignment,
            "concerns": [],
            "suggestions": [],
            "trust_update": 0.0,
        }

        if ai_confidence > 0.9 and self.decision_style == "conservative":
            evaluation["concerns"].append(
                "AI confidence seems too high for this situation"
            )
            evaluation["acceptance_level"] = "cautious"
        elif ai_confidence < 0.3 and self.decision_style == "aggressive":
            evaluation["concerns"].append("AI seems too conservative")
            evaluation["suggestions"].append(
                "Consider more aggressive pricing"
            )
            evaluation["acceptance_level"] = "skeptical"

        if alignment > 0.8:
            evaluation["trust_update"] = 0.05
        elif alignment < 0.3:
            evaluation["trust_update"] = -0.03
        return evaluation


@dataclass
class AIExplanation:
    """Structured explanation accompanying an AI recommendation."""

    primary_factors: List[str]
    secondary_factors: List[str]
    data_sources: List[str]
    reasoning_steps: List[str]
    confidence_breakdown: Dict[str, float]
    alternative_considered: List[Dict]
    uncertainty_sources: List[str]


class ExplainableAI:
    """Rule-based recommendation system with an explanation record."""

    def __init__(self, ai_id: str, specialty_areas: List[str]):
        self.ai_id = ai_id
        self.specialty_areas = specialty_areas
        self.decision_history = []
        self.performance_metrics = defaultdict(list)
        self.learned_patterns = {}
        self.confidence_calibration = {}

    def generate_pricing_recommendation(
        self, situation: Dict
    ) -> Tuple[Dict, AIExplanation]:
        """Analyze a situation and return a price recommendation and rationale."""
        analysis = self._analyze_situation(situation)
        recommendation = self._generate_recommendation(analysis, situation)
        explanation = self._create_explanation(
            analysis, recommendation, situation
        )
        self.decision_history.append(recommendation.copy())
        return recommendation, explanation

    def _analyze_situation(self, situation: Dict) -> Dict:
        """Classify market pressure and demand trend using stated rules."""
        analysis = {
            "market_conditions": "stable",
            "competitive_pressure": "medium",
            "demand_trend": situation.get("demand_trend", "stable"),
            "capacity_utilization": 0.75,
            "historical_patterns": [],
            "risk_factors": [],
            "opportunity_factors": [],
        }

        sensitivity = situation.get("price_sensitivity")
        if sensitivity is not None and sensitivity > 0.02:
            analysis["competitive_pressure"] = "high"
            analysis["risk_factors"].append("High price sensitivity")
        elif sensitivity is not None and sensitivity < 0.005:
            analysis["competitive_pressure"] = "low"
            analysis["opportunity_factors"].append("Low price sensitivity")

        if analysis["demand_trend"] == "increasing":
            analysis["opportunity_factors"].append("Growing demand")
        elif analysis["demand_trend"] == "decreasing":
            analysis["risk_factors"].append("Declining demand")
        return analysis

    def _generate_recommendation(self, analysis: Dict, situation: Dict) -> Dict:
        """Apply transparent price-adjustment rules to a base price."""
        base_price = situation.get("base_price", 100.0)
        adjustment = 0.0
        confidence = 0.7

        if analysis["competitive_pressure"] == "high":
            adjustment -= 5.0
            confidence -= 0.1
        elif analysis["competitive_pressure"] == "low":
            adjustment += 8.0
            confidence += 0.1

        if analysis["demand_trend"] == "increasing":
            adjustment += 3.0
            confidence += 0.05
        elif analysis["demand_trend"] == "decreasing":
            adjustment -= 2.0
            confidence -= 0.05

        price = base_price + adjustment
        confidence = float(np.clip(confidence, 0.2, 1.0))
        reasoning = []
        if adjustment > 0:
            reasoning.append(
                "Premium pricing reflects the stated market and demand rules"
            )
        elif adjustment < 0:
            reasoning.append(
                "Competitive pricing reflects the stated market and demand rules"
            )

        return {
            "recommended_price": price,
            "confidence": confidence,
            "expected_outcome": {
                "market_share": None,
                "profit_margin": None,
                "risk_level": "medium",
            },
            "reasoning_summary": "; ".join(reasoning),
            "alternatives": [],
        }

    def _create_explanation(
        self, analysis: Dict, recommendation: Dict, situation: Dict
    ) -> AIExplanation:
        """Expose the rule inputs, factors, and illustrative uncertainties."""
        primary_factors = []
        if analysis["competitive_pressure"] != "medium":
            primary_factors.append(
                f"Competitive pressure: {analysis['competitive_pressure']}"
            )
        if analysis["demand_trend"] != "stable":
            primary_factors.append(
                f"Demand trend: {analysis['demand_trend']}"
            )

        alternatives = [
            {
                "price": recommendation["recommended_price"] - 5,
                "expected_outcome": "Illustrative lower-price alternative",
                "reason_for_rejection": "Not evaluated by the rule-based model",
            },
            {
                "price": recommendation["recommended_price"] + 5,
                "expected_outcome": "Illustrative higher-price alternative",
                "reason_for_rejection": "Not evaluated by the rule-based model",
            },
        ]
        uncertainty_sources = []
        if analysis["competitive_pressure"] == "high":
            uncertainty_sources.append("Competitor response uncertainty")
        if analysis["demand_trend"] != "stable":
            uncertainty_sources.append("Demand forecast uncertainty")

        return AIExplanation(
            primary_factors=primary_factors,
            secondary_factors=(
                analysis["risk_factors"] + analysis["opportunity_factors"]
            ),
            data_sources=["Situation fields supplied to the rule-based model"],
            reasoning_steps=[
                "Classify price sensitivity and demand trend using stated thresholds",
                "Apply the corresponding price adjustments to the base price",
                "Clip the confidence score to its defined range",
            ],
            confidence_breakdown={
                "data_quality": 0.8,
                "model_accuracy": 0.7,
                "market_stability": 0.6,
                "pattern_recognition": 0.75,
            },
            alternative_considered=alternatives,
            uncertainty_sources=uncertainty_sources,
        )

    def learn_from_feedback(self, human_feedback: Dict, actual_outcome: Dict):
        """Record feedback and update the observed success rate by situation type."""
        predicted = human_feedback.get("ai_confidence", 0.5)
        assessed = human_feedback.get("human_confidence", 0.5)
        error = abs(predicted - assessed)
        self.performance_metrics["confidence_calibration"].append(error)

        situation_type = human_feedback.get("situation_type")
        if situation_type is None:
            return
        pattern = self.learned_patterns.setdefault(
            situation_type, {"success_rate": 0.0, "examples": []}
        )
        pattern["examples"].append(
            {"feedback": human_feedback, "outcome": actual_outcome}
        )
        successes = sum(
            bool(example["outcome"].get("success", False))
            for example in pattern["examples"]
        )
        pattern["success_rate"] = successes / len(pattern["examples"])

    def get_performance_summary(self) -> Dict:
        """Summarize recorded decisions, calibration, and learned patterns."""
        confidences = [
            decision.get("confidence", 0.5)
            for decision in self.decision_history
        ]
        errors = self.performance_metrics["confidence_calibration"]
        return {
            "total_decisions": len(self.decision_history),
            "average_confidence": float(np.mean(confidences)) if confidences else 0.0,
            "confidence_calibration_error": float(np.mean(errors)) if errors else 0.0,
            "learned_patterns": len(self.learned_patterns),
            "specialty_areas": self.specialty_areas,
        }

In [ ]:
class HumanAIPartnership:
    """Coordinate AI recommendations, human judgment, and market outcomes."""

    def __init__(
        self,
        ports: List[Port],
        shipping_lines: List[ShippingLine],
        market_params: MarketParameters,
    ):
        if not ports or not shipping_lines:
            raise ValueError("At least one port and one shipping line are required.")
        if any(len(port.costs) != len(shipping_lines) for port in ports):
            raise ValueError("Each port must have one cost per shipping line.")
        if market_params.min_price > market_params.max_price:
            raise ValueError("The minimum price cannot exceed the maximum price.")
        if market_params.price_sensitivity < 0:
            raise ValueError("Price sensitivity must be non-negative.")

        self.ports = ports
        self.shipping_lines = shipping_lines
        self.market_params = market_params
        self.num_ports = len(ports)
        self.num_shipping_lines = len(shipping_lines)
        self.human_managers = [
            HumanManager(
                id=0,
                name="Sarah Chen",
                experience_years=15,
                expertise_areas=[
                    "strategic_planning",
                    "relationship_management",
                    "risk_assessment",
                ],
                decision_style="balanced",
            ),
            HumanManager(
                id=1,
                name="Michael Rodriguez",
                experience_years=12,
                expertise_areas=[
                    "market_analysis",
                    "competitive_intelligence",
                    "pricing_strategy",
                ],
                decision_style="aggressive",
            ),
            HumanManager(
                id=2,
                name="Emma Thompson",
                experience_years=8,
                expertise_areas=[
                    "operations",
                    "capacity_planning",
                    "efficiency_optimization",
                ],
                decision_style="conservative",
            ),
        ]
        self.ai_systems = [
            ExplainableAI(
                ai_id="AI_PRICING_1",
                specialty_areas=[
                    "pricing_optimization",
                    "market_analysis",
                    "demand_forecasting",
                ],
            ),
            ExplainableAI(
                ai_id="AI_STRATEGY_1",
                specialty_areas=[
                    "strategic_planning",
                    "competitive_intelligence",
                    "risk_assessment",
                ],
            ),
            ExplainableAI(
                ai_id="AI_OPERATIONS_1",
                specialty_areas=[
                    "capacity_optimization",
                    "operational_efficiency",
                    "cost_analysis",
                ],
            ),
        ]
        self.partnership_metrics = {
            "trust_development": defaultdict(list),
            "decision_quality": [],
            "collaboration_score": [],
            "learning_progress": [],
            "decision_outcomes": [],
        }
        self.decision_history = []
        self.current_prices = self._initialize_prices()
        self.market_shares = np.zeros(
            (self.num_ports, self.num_shipping_lines), dtype=float
        )
        self.update_market_state()

    def _initialize_prices(self) -> np.ndarray:
        """Draw initial prices within the market bounds."""
        return np.random.uniform(
            self.market_params.min_price,
            self.market_params.max_price,
            (self.num_ports, self.num_shipping_lines),
        )

    def calculate_market_share(self, port_idx: int, shipping_line_idx: int) -> float:
        """Return the logit-model demand allocated to one port and line."""
        beta = self.market_params.price_sensitivity
        demand = self.shipping_lines[shipping_line_idx].demand
        prices = self.current_prices[:, shipping_line_idx]
        utilities = np.exp(-beta * prices)
        total_utility = np.sum(utilities)
        if total_utility <= 0:
            return 0.0
        return float(demand * utilities[port_idx] / total_utility)

    def update_market_state(self):
        """Refresh all port-by-line demand allocations."""
        for port_idx in range(self.num_ports):
            for line_idx in range(self.num_shipping_lines):
                self.market_shares[port_idx, line_idx] = self.calculate_market_share(
                    port_idx, line_idx
                )

    def collaborative_pricing_decision(
        self, port_idx: int, shipping_line_idx: int
    ) -> Dict:
        """Collect AI and human inputs, then apply the consensus rule."""
        situation = {
            "port_id": port_idx,
            "shipping_line_id": shipping_line_idx,
            "current_price": self.current_prices[port_idx, shipping_line_idx],
            "base_price": self.ports[port_idx].costs[shipping_line_idx] + 20,
            "price_sensitivity": self.market_params.price_sensitivity,
            "demand_trend": "stable",
            "relationship_strength": self.shipping_lines[
                shipping_line_idx
            ].relationship_strength,
            "strategic_importance": self.shipping_lines[
                shipping_line_idx
            ].strategic_importance,
        }

        ai_recommendations = []
        for ai in self.ai_systems:
            recommendation, explanation = ai.generate_pricing_recommendation(
                situation
            )
            ai_recommendations.append(
                {
                    "ai_id": ai.ai_id,
                    "recommendation": recommendation,
                    "explanation": explanation,
                }
            )

        human_insights = []
        for human in self.human_managers:
            human_insights.append(
                {
                    "human_id": human.id,
                    "human_name": human.name,
                    "insight": human.provide_strategic_insight(situation),
                }
            )
        return self._facilitate_collaboration(
            situation, ai_recommendations, human_insights
        )

    def _facilitate_collaboration(
        self, situation: Dict, ai_recommendations: List[Dict], human_insights: List[Dict]
    ) -> Dict:
        """Record the discussion and calculate a consensus decision."""
        collaboration_log = {
            "situation": situation,
            "ai_recommendations": ai_recommendations,
            "human_insights": human_insights,
            "discussion_points": [],
            "consensus_level": 0.0,
            "final_decision": None,
            "decision_rationale": "",
            "confidence_score": 0.0,
        }

        for ai_rec in ai_recommendations:
            recommendation = ai_rec["recommendation"]
            collaboration_log["discussion_points"].append(
                {
                    "speaker": ai_rec["ai_id"],
                    "type": "ai_recommendation",
                    "content": (
                        f"Recommend price ${recommendation['recommended_price']:.2f} "
                        f"with {recommendation['confidence']:.1%} confidence"
                    ),
                    "reasoning": recommendation["reasoning_summary"],
                    "explanation": ai_rec["explanation"],
                }
            )

        for human_entry in human_insights:
            insight = human_entry["insight"]
            collaboration_log["discussion_points"].append(
                {
                    "speaker": human_entry["human_name"],
                    "type": "human_insight",
                    "content": ", ".join(insight["strategic_factors"]),
                    "reasoning": ", ".join(
                        insight["relationship_considerations"]
                    ),
                    "confidence": insight["confidence"],
                }
            )

        for human in self.human_managers:
            for ai_rec in ai_recommendations:
                evaluation = human.evaluate_ai_recommendation(
                    ai_rec["recommendation"]
                )
                collaboration_log["discussion_points"].append(
                    {
                        "speaker": human.name,
                        "type": "human_evaluation",
                        "content": (
                            f"Evaluation of {ai_rec['ai_id']} recommendation: "
                            f"{evaluation['acceptance_level']}"
                        ),
                        "concerns": evaluation["concerns"],
                        "suggestions": evaluation["suggestions"],
                    }
                )
                human.trust_in_ai = float(
                    np.clip(human.trust_in_ai + evaluation["trust_update"], 0, 1)
                )

        final_decision = self._reach_consensus(
            ai_recommendations, human_insights, collaboration_log
        )
        confidence = self._calculate_confidence_score(
            final_decision, ai_recommendations, human_insights
        )
        consensus = self._calculate_consensus_level(
            ai_recommendations, human_insights, final_decision
        )
        collaboration_log["final_decision"] = final_decision
        collaboration_log["confidence_score"] = confidence
        collaboration_log["consensus_level"] = consensus
        return collaboration_log

    def _reach_consensus(
        self, ai_recommendations: List[Dict], human_insights: List[Dict],
        collaboration_log: Dict
    ) -> Dict:
        """Combine confidence-weighted AI prices with rule-based human adjustments."""
        ai_prices = [
            entry["recommendation"]["recommended_price"]
            for entry in ai_recommendations
        ]
        ai_confidences = [
            entry["recommendation"]["confidence"]
            for entry in ai_recommendations
        ]
        total_confidence = sum(ai_confidences)
        if total_confidence > 0:
            ai_weighted_price = sum(
                price * confidence
                for price, confidence in zip(ai_prices, ai_confidences)
            ) / total_confidence
        else:
            ai_weighted_price = float(np.mean(ai_prices))

        human_adjustment = 0.0
        for entry in human_insights:
            factors = entry["insight"]["strategic_factors"]
            if any("High-value customer" in factor for factor in factors):
                human_adjustment += 2.0
            if any("Long-term partnership" in factor for factor in factors):
                human_adjustment -= 1.0
            if any("Opportunistic business" in factor for factor in factors):
                human_adjustment += 3.0

        proposed_price = ai_weighted_price + human_adjustment
        price = float(
            np.clip(
                proposed_price,
                self.market_params.min_price,
                self.market_params.max_price,
            )
        )
        rationale = [f"AI-weighted price: ${ai_weighted_price:.2f}"]
        if human_adjustment != 0:
            rationale.append(f"Human strategic adjustment: ${human_adjustment:+.2f}")
        if not np.isclose(price, proposed_price):
            rationale.append(f"Price clipped to market bounds: ${price:.2f}")
        rationale.append(f"Final collaborative price: ${price:.2f}")
        return {
            "price": price,
            "rationale": "; ".join(rationale),
            "ai_contribution": ai_weighted_price,
            "human_contribution": human_adjustment,
            "decision_type": "collaborative",
        }

    def _calculate_confidence_score(
        self, final_decision: Dict, ai_recommendations: List[Dict],
        human_insights: List[Dict]
    ) -> float:
        """Combine AI confidence, human confidence, trust, and AI price agreement."""
        ai_confidences = [
            entry["recommendation"]["confidence"]
            for entry in ai_recommendations
        ]
        human_confidences = [
            entry["insight"]["confidence"] for entry in human_insights
        ]
        avg_trust = np.mean([human.trust_in_ai for human in self.human_managers])
        ai_prices = [
            entry["recommendation"]["recommended_price"]
            for entry in ai_recommendations
        ]
        price_variance = float(np.var(ai_prices))
        agreement = 1.0 - min(price_variance / 100.0, 1.0)
        score = (
            0.3 * np.mean(ai_confidences)
            + 0.3 * np.mean(human_confidences)
            + 0.2 * avg_trust
            + 0.2 * agreement
        )
        return float(np.clip(score, 0.0, 1.0))

    def _calculate_consensus_level(
        self, ai_recommendations: List[Dict], human_insights: List[Dict],
        final_decision: Dict
    ) -> float:
        """Summarize AI price agreement and an explicitly illustrative human score."""
        ai_prices = np.array(
            [entry["recommendation"]["recommended_price"]
             for entry in ai_recommendations],
            dtype=float,
        )
        mean_price = float(np.mean(ai_prices))
        if np.isclose(mean_price, 0.0):
            ai_consensus = 0.0
        else:
            ai_consensus = float(
                np.clip(1.0 - np.std(ai_prices) / abs(mean_price), 0.0, 1.0)
            )
        human_consensus = 0.7
        return float((ai_consensus + human_consensus) / 2.0)

    def implement_decision(
        self, port_idx: int, shipping_line_idx: int, decision: Dict,
        confidence_score: float = 0.7
    ) -> Dict:
        """Apply a price, record the pre-decision state, and calculate outcomes."""
        prices_before = self.current_prices.copy()
        market_before = self.market_shares.copy()
        self.current_prices[port_idx, shipping_line_idx] = decision["price"]
        self.update_market_state()
        record = {
            "timestamp": time.time(),
            "port_idx": port_idx,
            "shipping_line_idx": shipping_line_idx,
            "decision": decision.copy(),
            "confidence_score": confidence_score,
            "market_state_before": market_before,
            "prices_before": prices_before,
        }
        self.decision_history.append(record)
        outcomes = self._calculate_decision_outcomes(record)
        self._learn_from_decision(record, outcomes)
        return outcomes

    def _calculate_decision_outcomes(self, decision_record: Dict) -> Dict:
        """Calculate profit, margin, allocation change, and the stated success flag."""
        port_idx = decision_record["port_idx"]
        line_idx = decision_record["shipping_line_idx"]
        price = decision_record["decision"]["price"]
        cost = self.ports[port_idx].costs[line_idx]
        market_share = self.market_shares[port_idx, line_idx]
        profit = (price - cost) * market_share
        margin = (price - cost) / price if price > 0 else 0.0
        old_share = decision_record["market_state_before"][port_idx, line_idx]
        share_change = market_share - old_share
        return {
            "profit": float(profit),
            "profit_margin": float(margin),
            "market_share": float(market_share),
            "market_share_change": float(share_change),
            "price": float(price),
            "success": bool(profit > 0 and share_change >= -0.1),
        }

    def _learn_from_decision(self, decision_record: Dict, outcomes: Dict):
        """Pass observed outcomes to the illustrative AI and trust updates."""
        line_idx = decision_record["shipping_line_idx"]
        for ai in self.ai_systems:
            feedback = {
                "ai_confidence": 0.7,
                "human_confidence": decision_record["confidence_score"],
                "situation_type": (
                    f"port_{decision_record['port_idx']}_sl_{line_idx}"
                ),
            }
            ai.learn_from_feedback(feedback, outcomes)

        trust_change = 0.02 if outcomes["success"] else -0.01
        for human in self.human_managers:
            human.trust_in_ai = float(
                np.clip(human.trust_in_ai + trust_change, 0.0, 1.0)
            )
        self.partnership_metrics["decision_outcomes"].append(outcomes)
        trust_levels = [human.trust_in_ai for human in self.human_managers]
        self.partnership_metrics["trust_development"]["overall"].append(
            float(np.mean(trust_levels))
        )
        for human in self.human_managers:
            self.partnership_metrics["trust_development"][human.name].append(
                human.trust_in_ai
            )

    def run_partnership_simulation(self, num_decisions: int = 50) -> Dict:
        """Run repeated random port–shipping-line collaborative decisions."""
        if num_decisions <= 0:
            raise ValueError("num_decisions must be positive.")
        print("\nHuman-AI Symbiotic Partnership Simulation")
        print(f"Number of decisions: {num_decisions}")
        print("=" * 60)

        results = {
            "decisions": [],
            "outcomes": [],
            "trust_evolution": defaultdict(list),
            "initial_trust": {
                human.name: human.trust_in_ai for human in self.human_managers
            },
        }
        for decision_num in range(num_decisions):
            port_idx = random.randint(0, self.num_ports - 1)
            line_idx = random.randint(0, self.num_shipping_lines - 1)
            log = self.collaborative_pricing_decision(port_idx, line_idx)
            outcomes = self.implement_decision(
                port_idx,
                line_idx,
                log["final_decision"],
                log["confidence_score"],
            )
            results["decisions"].append(log)
            results["outcomes"].append(outcomes)
            for human in self.human_managers:
                results["trust_evolution"][human.name].append(human.trust_in_ai)

            if decision_num % 10 == 0 or decision_num == num_decisions - 1:
                avg_trust = np.mean(
                    [human.trust_in_ai for human in self.human_managers]
                )
                total_profit = sum(item["profit"] for item in results["outcomes"])
                success_rate = np.mean(
                    [item["success"] for item in results["outcomes"]]
                )
                print(
                    f"Decision {decision_num}: Trust={avg_trust:.1%}, "
                    f"Profit=${total_profit:,.0f}, Success={success_rate:.1%}"
                )
        return results

In [ ]:
def create_concrete_example():
    """Create the three-port, three-shipping-line teaching example."""
    ports = [
        Port(id=0, name="Port A", capacity=20000, costs=[80, 85, 82]),
        Port(id=1, name="Port B", capacity=18000, costs=[85, 78, 88]),
        Port(id=2, name="Port C", capacity=22000, costs=[75, 90, 80]),
    ]
    shipping_lines = [
        ShippingLine(
            id=0, name="Global Shipping Line", demand=12000, priority=1,
            relationship_strength=0.9, strategic_importance=0.95,
        ),
        ShippingLine(
            id=1, name="Pacific Trade Line", demand=15000, priority=2,
            relationship_strength=0.7, strategic_importance=0.6,
        ),
        ShippingLine(
            id=2, name="Atlantic Container Line", demand=10000, priority=3,
            relationship_strength=0.4, strategic_importance=0.3,
        ),
    ]
    market_params = MarketParameters(
        price_sensitivity=0.01,
        min_price=80,
        max_price=150,
    )
    return ports, shipping_lines, market_params


ports, shipping_lines, market_params = create_concrete_example()
partnership = HumanAIPartnership(ports, shipping_lines, market_params)

print("Human-AI Partnership Configuration:")
print(f"Number of ports: {len(ports)}")
print(f"Number of shipping lines: {len(shipping_lines)}")
print(f"Total market demand: {sum(line.demand for line in shipping_lines):,} TEU")
print(f"Total port capacity: {sum(port.capacity for port in ports):,} TEU")
print("\nHuman managers:")
for human in partnership.human_managers:
    print(
        f"- {human.name}: {human.experience_years} years of experience, "
        f"{human.decision_style} style"
    )
    print(f"  Expertise: {', '.join(human.expertise_areas)}")
print("\nAI systems:")
for ai in partnership.ai_systems:
    print(f"- {ai.ai_id}: {', '.join(ai.specialty_areas)}")
print("\nShipping-line relationships:")
for line in shipping_lines:
    print(
        f"- {line.name}: Relationship {line.relationship_strength:.1%}, "
        f"Strategic importance {line.strategic_importance:.1%}"
    )

# The logit model allocates each line's demand across ports; capacity is descriptive only.
for line_idx, line in enumerate(shipping_lines):
    allocated_demand = partnership.market_shares[:, line_idx].sum()
    assert np.isclose(allocated_demand, line.demand, rtol=1e-10, atol=1e-8)

## One collaborative pricing decision

The demonstration below selects Port A and the highest-priority shipping line. The code prints each AI recommendation and its explanation, the managers' strategic inputs, the consensus price, and the modeled outcome. The starting port prices are drawn randomly within the market bounds, so the market-share outcome can vary between fresh runs.

The price recommendation is not obtained by maximizing the profit equation. Rather, this section follows the notebook's stated collaboration rule: confidence-weighted AI recommendations plus human strategic adjustments, followed by clipping to the allowable price interval.

In [ ]:
def demonstrate_collaborative_decision():
    """Display and implement one collaborative price decision."""
    print("\n" + "=" * 80)
    print("COLLABORATIVE PRICING DECISION DEMONSTRATION")
    print("=" * 80)
    port_idx = 0
    line_idx = 0
    line = shipping_lines[line_idx]
    print(
        f"Scenario: {ports[port_idx].name} and {line.name} "
        f"(priority {line.priority})"
    )
    print(f"Relationship strength: {line.relationship_strength:.1%}")
    print(f"Strategic importance: {line.strategic_importance:.1%}")

    log = partnership.collaborative_pricing_decision(port_idx, line_idx)
    print("\nAI recommendations:")
    for entry in log["ai_recommendations"]:
        recommendation = entry["recommendation"]
        print(f"\n{entry['ai_id']}:")
        print(f"  Recommended price: ${recommendation['recommended_price']:.2f}")
        print(f"  Confidence: {recommendation['confidence']:.1%}")
        print(f"  Reasoning: {recommendation['reasoning_summary']}")
        explanation = entry["explanation"]
        if explanation.primary_factors:
            print(f"  Primary factors: {', '.join(explanation.primary_factors)}")
        if explanation.uncertainty_sources:
            print(
                "  Uncertainty sources: "
                f"{', '.join(explanation.uncertainty_sources)}"
            )

    print("\nHuman strategic insights:")
    for entry in log["human_insights"]:
        insight = entry["insight"]
        manager = partnership.human_managers[entry["human_id"]]
        print(f"\n{entry['human_name']} ({manager.experience_years} years):")
        print(f"  Strategic factors: {', '.join(insight['strategic_factors'])}")
        print(
            "  Relationship considerations: "
            f"{', '.join(insight['relationship_considerations'])}"
        )
        print(f"  Risk assessment: {insight['risk_assessment']}")
        print(f"  Confidence: {insight['confidence']:.1%}")

    final_decision = log["final_decision"]
    print("\nFinal collaborative decision:")
    print(f"Price: ${final_decision['price']:.2f}")
    print(f"Rationale: {final_decision['rationale']}")
    print(f"AI contribution: ${final_decision['ai_contribution']:.2f}")
    print(f"Human adjustment: ${final_decision['human_contribution']:+.2f}")
    print(f"Overall confidence: {log['confidence_score']:.1%}")
    print(f"Consensus level: {log['consensus_level']:.1%}")

    outcomes = partnership.implement_decision(
        port_idx, line_idx, final_decision, log["confidence_score"]
    )
    print("\nModeled decision outcomes:")
    print(f"Profit: ${outcomes['profit']:,.2f}")
    print(f"Profit margin: {outcomes['profit_margin']:.1%}")
    print(f"Allocated demand: {outcomes['market_share']:.0f} TEU")
    print(f"Allocation change: {outcomes['market_share_change']:+.0f} TEU")
    print(f"Success under the stated rule: {outcomes['success']}")
    return log, outcomes


demo_log, demo_outcomes = demonstrate_collaborative_decision()

## Repeated partnership simulation

Each simulation step samples a port and shipping line, makes a collaborative decision, updates the selected price, recalculates market allocation, and records outcomes. The success flag is defined in code as positive modeled profit and a market-share change no worse than the specified threshold. It should be interpreted as a simulation rule, not as a general measure of decision quality.

Trust changes are also rule-based: success increases each manager's trust by a fixed increment, while failure decreases it. Those changes do not represent a validated psychological or organizational model.

In [ ]:
def run_partnership_simulation():
    """Run the stated simulation and summarize its recorded metrics."""
    results = partnership.run_partnership_simulation(num_decisions=50)
    outcomes = results["outcomes"]
    decisions = results["decisions"]
    total_profit = sum(outcome["profit"] for outcome in outcomes)
    average_margin = float(np.mean([item["profit_margin"] for item in outcomes]))
    success_rate = float(np.mean([item["success"] for item in outcomes]))
    average_confidence = float(
        np.mean([item["confidence_score"] for item in decisions])
    )
    average_consensus = float(
        np.mean([item["consensus_level"] for item in decisions])
    )

    print("\nSimulation summary:")
    print(f"Total profit across decisions: ${total_profit:,.2f}")
    print(f"Mean decision profit margin: {average_margin:.1%}")
    print(f"Success rate under the stated rule: {success_rate:.1%}")
    print(f"Mean decision confidence: {average_confidence:.1%}")
    print(f"Mean consensus score: {average_consensus:.1%}")

    print("\nTrust development:")
    initial = results["initial_trust"]
    for name, history in results["trust_evolution"].items():
        final = history[-1]
        change = final - initial[name]
        print(f"{name}: {initial[name]:.1%} to {final:.1%} ({change:+.1%})")

    print("\nAI performance summary:")
    for ai in partnership.ai_systems:
        summary = ai.get_performance_summary()
        print(f"\n{ai.ai_id}:")
        print(f"  Recorded recommendations: {summary['total_decisions']}")
        print(f"  Mean recommendation confidence: {summary['average_confidence']:.1%}")
        print(
            "  Illustrative confidence calibration error: "
            f"{summary['confidence_calibration_error']:.3f}"
        )
        print(f"  Situation patterns recorded: {summary['learned_patterns']}")
    return results


simulation_results = run_partnership_simulation()

## Critical negotiation illustration

This scenario temporarily increases the selected shipping line's demand and strategic-importance value, then applies three explicitly specified decision rules: a human-only price, an AI-only price, and the collaborative price. To make the market-share comparison interpretable, each approach is evaluated from the same starting price vector. The illustrative partnership-impact factors below are assumptions applied to the raw profit margin; they are not learned from data and do not alter the logit demand allocation.

In [ ]:
def simulate_critical_negotiation():
    """Compare three rule-based decisions under a shared market-price baseline."""
    port_idx = 0
    line_idx = 0
    line = shipping_lines[line_idx]
    original_demand = line.demand
    original_importance = line.strategic_importance
    baseline_prices = partnership.current_prices.copy()
    original_market_shares = partnership.market_shares.copy()

    line.demand = original_demand * 1.5
    line.strategic_importance = 1.0
    situation = {
        "port_id": port_idx,
        "shipping_line_id": line_idx,
        "current_price": baseline_prices[port_idx, line_idx],
        "base_price": ports[port_idx].costs[line_idx] + 20,
        "price_sensitivity": market_params.price_sensitivity,
        "demand_trend": "increasing",
        "relationship_strength": line.relationship_strength,
        "strategic_importance": line.strategic_importance,
    }

    human = partnership.human_managers[0]
    human_insight = human.provide_strategic_insight(situation)
    human_price = situation["base_price"] + 5.0
    if human.decision_style == "aggressive":
        human_price += 10.0
    elif human.decision_style == "conservative":
        human_price -= 2.0

    ai = partnership.ai_systems[0]
    ai_recommendation, ai_explanation = ai.generate_pricing_recommendation(
        situation
    )
    collaborative_log = partnership.collaborative_pricing_decision(
        port_idx, line_idx
    )
    decisions = {
        "human_only": {"price": human_price},
        "ai_only": {"price": ai_recommendation["recommended_price"]},
        "collaborative": collaborative_log["final_decision"],
    }
    partnership_impacts = {
        "human_only": 0.05,
        "ai_only": -0.02,
        "collaborative": 0.15,
    }

    results = {}
    try:
        print("\nCritical negotiation illustration")
        print(f"Customer: {line.name}")
        print(f"Demand in scenario: {line.demand:,.0f} TEU")
        print("Partnership-impact factors are illustrative assumptions.")
        print(f"Human strategic factors: {', '.join(human_insight['strategic_factors'])}")
        print(
            "AI primary factors: "
            f"{', '.join(ai_explanation.primary_factors) or 'none under the stated rules'}"
        )

        for approach, decision in decisions.items():
            partnership.current_prices = baseline_prices.copy()
            partnership.current_prices[port_idx, line_idx] = decision["price"]
            partnership.update_market_state()
            price = float(decision["price"])
            cost = ports[port_idx].costs[line_idx]
            share = float(partnership.market_shares[port_idx, line_idx])
            raw_margin = (price - cost) / price if price > 0 else 0.0
            impact = partnership_impacts[approach]
            displayed_margin = raw_margin * (1.0 + impact)
            results[approach] = {
                "price": price,
                "profit": (price - cost) * share,
                "raw_profit_margin": raw_margin,
                "profit_margin_after_assumed_impact": displayed_margin,
                "market_share": share,
                "partnership_impact_assumption": impact,
            }
            print(
                f"{approach.replace('_', ' ').title()}: price=${price:.2f}, "
                f"allocated demand={share:.0f} TEU, "
                f"raw margin={raw_margin:.1%}, "
                f"margin after assumed factor={displayed_margin:.1%}"
            )

        comparison = pd.DataFrame(
            [
                {
                    "Approach": approach.replace("_", " ").title(),
                    "Price": result["price"],
                    "Raw Profit Margin": result["raw_profit_margin"],
                    "Profit Margin After Assumed Factor": result[
                        "profit_margin_after_assumed_impact"
                    ],
                    "Allocated Demand (TEU)": result["market_share"],
                    "Assumed Partnership Impact": result[
                        "partnership_impact_assumption"
                    ],
                }
                for approach, result in results.items()
            ]
        )
        print("\nScenario comparison:")
        print(comparison.round(3).to_string(index=False))
    finally:
        line.demand = original_demand
        line.strategic_importance = original_importance
        partnership.current_prices = baseline_prices
        partnership.market_shares = original_market_shares
    return results


negotiation_results = simulate_critical_negotiation()

## Visualizing the simulation

The plots summarize recorded simulation quantities. Trust and success are outcomes of the stated update rules; confidence and consensus are constructed scores, not independently validated measures. Profit is cumulative across the sequence of simulated decisions, so it should not be interpreted as a time-series forecast or as an isolated comparison with other methods.

In [ ]:
def visualize_partnership_results():
    """Plot trust, decision scores, profits, and negotiation assumptions."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("Human-AI Partnership Simulation", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    for name, history in simulation_results["trust_evolution"].items():
        ax.plot(range(1, len(history) + 1), history, label=name, linewidth=2)
    ax.set_title("Trust under the Simulation Update Rule")
    ax.set_xlabel("Decision number")
    ax.set_ylabel("Trust level (0–1)")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[0, 1]
    decisions = simulation_results["decisions"]
    ax.plot(
        [item["confidence_score"] for item in decisions],
        label="Confidence score",
    )
    ax.plot(
        [item["consensus_level"] for item in decisions],
        label="Consensus level",
    )
    ax.set_title("Constructed Decision Scores")
    ax.set_xlabel("Decision index")
    ax.set_ylabel("Score (0–1)")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[0, 2]
    outcomes = simulation_results["outcomes"]
    profits = [item["profit"] for item in outcomes]
    margins = [item["profit_margin"] for item in outcomes]
    ax.plot(np.cumsum(profits), color="green", label="Cumulative modeled profit")
    ax.set_title("Cumulative Modeled Profit")
    ax.set_xlabel("Decision index")
    ax.set_ylabel("Profit (currency units)")
    ax.grid(True, alpha=0.3)
    margin_axis = ax.twinx()
    margin_axis.plot(margins, "r--", alpha=0.7, label="Profit margin")
    margin_axis.set_ylabel("Profit margin")
    ax.legend(loc="upper left")
    margin_axis.legend(loc="upper right")

    ax = axes[1, 0]
    approaches = list(negotiation_results)
    displayed_margins = [
        negotiation_results[key]["profit_margin_after_assumed_impact"]
        for key in approaches
    ]
    bars = ax.bar(
        [key.replace("_", " ").title() for key in approaches],
        displayed_margins,
        color=["#1f77b4", "#ff7f0e", "#2ca02c"],
    )
    ax.set_title("Margin Including Scenario Assumptions")
    ax.set_ylabel("Adjusted illustrative margin")
    ax.tick_params(axis="x", rotation=30)
    for bar, value in zip(bars, displayed_margins):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1%}",
            ha="center",
            va="bottom",
        )

    ax = axes[1, 1]
    ai_names = [ai.ai_id for ai in partnership.ai_systems]
    recommendation_counts = [
        ai.get_performance_summary()["total_decisions"]
        for ai in partnership.ai_systems
    ]
    ax.bar(ai_names, recommendation_counts, color="#4c72b0")
    ax.set_title("Recorded AI Recommendations")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 2]
    window_size = 10
    success_rates = []
    for index in range(len(outcomes)):
        start = max(0, index - window_size + 1)
        window = outcomes[start:index + 1]
        success_rates.append(np.mean([item["success"] for item in window]))
    ax.plot(success_rates, color="purple", label="Rolling success rate")
    ax.axhline(0.8, color="red", linestyle="--", label="Illustrative 80% reference")
    ax.set_title("Rolling Success under the Stated Rule")
    ax.set_xlabel("Decision index")
    ax.set_ylabel("Success rate")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    plt.show()
    return fig


visualization_fig = visualize_partnership_results()

## Capability comparison

The following table compares the kinds of capabilities represented in this notebook with broad method categories. It is a qualitative description, not a benchmark: no comparable data for the other methods are supplied here, so numerical profit or success-rate rankings would not be supported. The partnership simulation itself is the only method evaluated by the executable example.

In [ ]:
def compare_method_capabilities():
    """Show a qualitative capability matrix without fabricated benchmarks."""
    comparison_data = [
        {
            "Method category": "Mathematical programming",
            "Human oversight": "Can be specified externally",
            "Explainability": "Model-dependent",
            "Relationship factors": "Must be modeled explicitly",
            "Adaptive learning": "Not inherent",
        },
        {
            "Method category": "Iterative strategic response",
            "Human oversight": "Can be included",
            "Explainability": "Rule-dependent",
            "Relationship factors": "Must be modeled explicitly",
            "Adaptive learning": "Not inherent",
        },
        {
            "Method category": "Evolutionary optimization",
            "Human oversight": "Can be included",
            "Explainability": "Often limited by search process",
            "Relationship factors": "Can be encoded in objectives",
            "Adaptive learning": "Not inherent",
        },
        {
            "Method category": "Deep reinforcement learning",
            "Human oversight": "Design-dependent",
            "Explainability": "Often requires additional tools",
            "Relationship factors": "Can be represented in state or reward",
            "Adaptive learning": "Possible with training",
        },
        {
            "Method category": "Human-AI partnership in this notebook",
            "Human oversight": "Explicit in the decision rule",
            "Explainability": "Rule-based explanations are displayed",
            "Relationship factors": "Used in illustrative adjustments",
            "Adaptive learning": "Illustrative feedback records",
        },
    ]
    comparison_df = pd.DataFrame(comparison_data)
    print("Qualitative capability matrix (not a performance benchmark):")
    print(comparison_df.to_string(index=False))
    return comparison_df


capability_comparison = compare_method_capabilities()

## Interpretation, limitations, and use

### What the example demonstrates

- AI recommendations can be accompanied by explicit inputs, rule steps, and uncertainty notes.
- Human strategic considerations can enter a transparent consensus rule.
- A logit choice model can translate port prices into a predicted allocation of shipping-line demand.
- Decision records can support descriptive summaries of profit, trust, confidence, and success under the chosen rules.

These are properties of the implemented simulation. They do not establish that human–AI collaboration outperforms either humans or AI alone in real port markets.

### Important limitations

- The AI systems use hand-written rules and fixed confidence components; they are not trained predictive models.
- Human insights are also hand-written rules, including a mapping from shipping-line index to strategic descriptions.
- The consensus and trust updates are illustrative, and trust is not used to weight the final price.
- Capacity is not enforced, and relationship strength does not directly affect demand or profit.
- Market share is calculated from a static logit model with price as its sole choice attribute.
- The negotiation scenario applies assumed partnership-impact multipliers to profit margins. Those factors are not measured outcomes, and the resulting adjusted margins should not be interpreted as empirical evidence.
- Random initial prices and random simulation selections are not seeded, so exact numerical results can differ between runs.

### When this modeling structure may be useful

A transparent collaboration workflow can be useful when managers need to inspect analytical recommendations and contribute contextual judgment. Before using such a system operationally, the assumptions, data, capacity constraints, decision criteria, confidence calibration, and outcome measures would need validation with appropriate domain evidence. This notebook's purpose is to illustrate the collaboration and accounting logic, not to provide an operational pricing recommendation.